# Faza 2 — AI-SPEAK preprocessing po VIPL demo pipeline-u


## Goal

Pretvori `spk*/ser/video_a/*.mp4` u numerisane `128×64` mouth JPEG frejmove
koristeći brzi BlazeFace detector, isti 68-point face alignment, VIPL afinu
transformaciju i crop. Ne generiše se manifest, vocab, split ili statičan ROI.
Neuspesi ostaju u običnom logu.


## Setup

Završeni `MyDrive/LipNet/ai_speak_lip.zip` već postoji. Smoke i puna BlazeFace
obrada su zato podrazumevano isključeni; notebook ostaje kao evidencija toka.


In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'face-alignment==1.4.1', 'editdistance>=0.8.1'],
    check=True,
)


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/nikolabakic/Vizuelno-prepoznavanje-govora-na-osnovu-pokreta-usana-pomo-u-LipNet-modela.git'
REPO = Path('/content/lipnet-serbian')
if not (REPO / 'lipnet').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
print('Repo:', REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import numpy as np
import torch
print(
    'GPU:', torch.cuda.get_device_name(0)
    if torch.cuda.is_available() else 'nije potreban za reuse postojećeg ZIP-a'
)


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/LipNet')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive izlaz:', DRIVE_ROOT)


## Steps

### 1. Kopiraj i raspakuj lokalni korpus


In [ ]:
import shutil
import zipfile

ZIP_ON_DRIVE = Path('/content/drive/MyDrive/processed.zip')  # promeni po potrebi
LOCAL_ZIP = Path('/content/processed.zip')
EXTRACT_ROOT = Path('/content/ai_speak_source')
OUTPUT_ROOT = Path('/content/ai_speak_lip_blazeface')
CHECKPOINT_DIR = DRIVE_ROOT / 'phase2_chunks_blazeface'
RUN_SMOKE_PREPROCESSING = False
RUN_FULL_PREPROCESSING = False
EXISTING_MOUTH_ARCHIVE = DRIVE_ROOT / 'ai_speak_lip.zip'
assert EXISTING_MOUTH_ARCHIVE.exists(), EXISTING_MOUTH_ARCHIVE
print('Zamrznuti mouth artefakt:', EXISTING_MOUTH_ARCHIVE)

CORPUS_ROOT = None
videos = []
annotations = []
if RUN_SMOKE_PREPROCESSING or RUN_FULL_PREPROCESSING:
    assert ZIP_ON_DRIVE.exists(), ZIP_ON_DRIVE
    if not LOCAL_ZIP.exists() or not zipfile.is_zipfile(LOCAL_ZIP):
        if LOCAL_ZIP.exists():
            LOCAL_ZIP.unlink()
        shutil.copy2(ZIP_ON_DRIVE, LOCAL_ZIP)
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    alignment = next(EXTRACT_ROOT.rglob('spk*/alignment/*.align'), None)
    if alignment is None:
        with zipfile.ZipFile(LOCAL_ZIP) as archive:
            archive.extractall(EXTRACT_ROOT)
        alignment = next(EXTRACT_ROOT.rglob('spk*/alignment/*.align'), None)
    assert alignment is not None
    CORPUS_ROOT = alignment.parents[2]
    videos = list(CORPUS_ROOT.glob('spk*/ser/video_a/*.mp4'))
    annotations = list(CORPUS_ROOT.glob('spk*/alignment/*.align'))
    assert videos and len(videos) == len(annotations)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    print('Korpus MP4/ALIGN:', len(videos), len(annotations))
else:
    print('processed.zip se ne kopira niti raspakuje.')


### 2. Prvo uradi jedan GPU smoke primer i upstream `_load_vid` proveru


In [ ]:
SMOKE_ROOT = Path('/content/ai_speak_lip_blazeface_smoke')
if RUN_SMOKE_PREPROCESSING:
    smoke_command = [
        sys.executable, '-m', 'scripts.prepare_ai_speak',
        '--corpus', str(CORPUS_ROOT), '--output', str(SMOKE_ROOT),
        '--device', 'cuda', '--face-detector', 'blazeface', '--limit', '1',
    ]
    smoke = subprocess.run(smoke_command, text=True, capture_output=True)
    print(smoke.stdout)
    if smoke.returncode:
        print(smoke.stderr)
        raise RuntimeError(f'Smoke preprocessing nije uspeo: {smoke.returncode}')
    from lipnet.dataset import MyDataset
    sample_folder = next(SMOKE_ROOT.glob('spk*/video/video_a/*'))
    sample_array = MyDataset._load_vid(sample_folder)
    normalized = sample_array / 255.0
    assert sample_array.shape[1:] == (64, 128, 3)
    assert 0.0 <= float(normalized.min()) <= float(normalized.max()) <= 1.0
    print('VIPL _load_vid:', sample_array.shape, 'range:', normalized.min(), normalized.max())
else:
    print('RUN_SMOKE_PREPROCESSING=False: BlazeFace smoke je preskočen.')


### 3. Obradi ceo korpus na GPU-u i nastavi bez ponavljanja gotovih klipova


In [ ]:
if RUN_FULL_PREPROCESSING:
    subprocess.run([
        sys.executable, '-m', 'scripts.prepare_ai_speak',
        '--corpus', str(CORPUS_ROOT), '--output', str(OUTPUT_ROOT),
        '--device', 'cuda', '--face-detector', 'blazeface', '--resume',
        '--report-every', '5',
        '--checkpoint-dir', str(CHECKPOINT_DIR),
        '--checkpoint-every', '10',
    ], check=True)
else:
    print('RUN_FULL_PREPROCESSING=False: ceo GPU posao je namerno preskočen.')


## Checks

### 4. Proveri potpunost i vizuelno pregledaj granične klipove


In [ ]:
assert EXISTING_MOUTH_ARCHIVE.stat().st_size > 0
if RUN_FULL_PREPROCESSING:
    from IPython.display import Image, display
    QA_PATH = OUTPUT_ROOT / 'qa_mouth_crops.jpg'
    FAILURE_LOG = OUTPUT_ROOT / 'failed_clips.log'
    PREPROCESSING_LOG = OUTPUT_ROOT / 'preprocessing.jsonl'
    assert QA_PATH.exists() and FAILURE_LOG.exists() and PREPROCESSING_LOG.exists()
    display(Image(filename=str(QA_PATH)))
    print('Nova obrada zahteva ručnu QA potvrdu pre arhiviranja.')
else:
    print('Postojeći ai_speak_lip.zip je potvrđen; QA i BlazeFace se ne ponavljaju.')


### 5. Arhiviraj JPEG foldere i logove na Drive


In [ ]:
if RUN_FULL_PREPROCESSING:
    MANUAL_QA_PASSED = False  # samo za namerno novu punu obradu
    assert MANUAL_QA_PASSED, 'Ručno pregledaj QA sliku pre nove arhive.'
    archive_base = Path('/content/ai_speak_lip')
    archive = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_ROOT))
    drive_archive = DRIVE_ROOT / 'ai_speak_lip.zip'
    shutil.copy2(archive, drive_archive)
    print('Sačuvano:', drive_archive, f'{drive_archive.stat().st_size/1024**3:.2f} GiB')
else:
    print('Postojeći ai_speak_lip.zip ostaje neizmenjen.')


## Next Steps

U nastavku projekta ne pokretati preprocessing ponovo. Faze 3–7 samo raspakuju
postojeći `ai_speak_lip.zip`; logovi nisu ulaz u trening.
